# 第09課 - 元認知設計模式


## 設定

此筆記本示範如何使用 Microsoft Agent Framework 來實作 Metacognition 設計模式。

**先決條件:**
- 已透過環境變數配置 Azure OpenAI 部署
- 已使用 Azure CLI 完成認證 (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 甚麼是元認知？

元認知是 **對思考的思考**。在 AI 代理人的範疇內，它意味著建立能夠：

- **自我反思** 其自身的輸出與推理過程
- **偵測錯誤** 並優雅地恢復，而不是默默失敗
- **評估** 他們的回應是否完整且有幫助
- **調整** 當初始方法不起作用時改變策略（例如，退回到備用系統）

元認知代理人不僅僅是回答問題 — 它監察自己的表現並即時調整。


## 主要與備用工具

一個常見的元認知模式是 **回退策略**。代理會先嘗試使用主要工具；如果失敗（例如 404 錯誤），代理會識別該失敗並透明地切換到備用工具。

這反映了真實世界中系統的情況，主要服務可能會無法使用，代理必須自行診斷問題，然後再選擇替代路徑。

以下我們定義兩個航班查詢工具：
- **Primary** — 覆蓋巴黎、東京及巴塞隆拿
- **Backup** — 覆蓋柏林、雪梨及紐約市


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## 具自我反思與錯誤復原能力的代理

下面的代理被指示先嘗試主要飛行系統、識別故障，並透明地回退至備用系統。每次回應後，它會簡短地自我評估是否已完全回答用戶的問題。


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## 自我評估模式

元認知的另一個面向是 **自我評估**：由一個獨立的代理人（或同一個代理人在第二次檢視時）檢視回應的完整性、準確性與有用性。

下面我們建立一個名為 `ResponseEvaluator` 的代理人，對旅行代理的回應在三個維度上進行評分。


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## 摘要

在這課中你學會了如何使用 Microsoft Agent Framework 建構 **元認知代理人**:

- **自我反思**: 監察自身的推理並透明地說明發生了什麼的代理人。
- **錯誤復原與備援**: 一種主工具 + 備份工具的模式，代理人會偵測失敗（例如 404 錯誤）並自動嘗試替代來源。
- **自我評估**: 一個獨立的評估代理人，為回應在完整性、準確性和有用性方面進行評分。

這些模式讓代理人更穩健、透明且值得信賴 — 對於投入生產部署而言是關鍵特性。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責聲明：
本文件已使用 AI 翻譯服務 Co-op Translator（https://github.com/Azure/co-op-translator）進行翻譯。雖然我們致力確保準確性，但自動翻譯可能包含錯誤或不準確之處。原文應被視為具權威性的資料來源。對於重要資訊，建議採用專業人工翻譯。我們不會對任何因使用本翻譯而導致的誤解或錯誤解讀承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
